In [ ]:
# =============================================================================
# jobs/27_fw_timing_validation.py
# Was each flood warning issued in time? For every gauge with thresholds, compare when the readings
# crossed the RES threshold with when the flood warning for its area was issued, against the target lead time.
#
# RUNS IN THE RESTRICTED DEPLOYMENT. It reads the thresholds (restricted schema) and writes its table there.
# It reads the telemetry mapping, the readings and the warning periods from the unrestricted schema. Nothing
# restricted is written to the unrestricted schema. Cadence: manual.
#
# THE IDEA. A gauge has an FW threshold, which triggers a flood warning, and a higher RES threshold, the level at
# which the impact is reached. The warning must be out before the reading first crosses RES, by at least the gauge's
# target lead time (hours, a column of the thresholds table). Rules and outcomes are in utils/fw_timing.py:
#   on time / too short / late / missed        for each upward crossing of RES
#   RES not reached / no readings                for a warning with no crossing while it was in force
# plus trigger_time and issue_delay_hours: how long the warning took after the FW threshold was crossed.
#
# THE LINKS. Thresholds -> gauge: FFFS_ID (compared without regard to case or leading zeros) and the parameter's family.
# Gauge -> readings: the matched lines of ea_rt_telemetry_mapping; where a gauge has several measures (datum or value
# type variants) the instantaneous one with the most readings is used. Gauge -> warning: the threshold row's warning
# area code, and the flood-warning-level periods of ea_warning_periods for that area.
#
# THE THRESHOLDS TABLE. Its column names are not fixed. Leave a widget blank to find the column by its usual names
# (see COLUMN_GUESSES in utils/fw_timing.py); the notebook prints what it found and the values of the type column, so a
# wrong guess is easy to see. Needed: FFFS_ID, a type column and a value column. Wanted: parameter, target lead time
# (hours) and warning area code.
#
# WIDGETS
#   fffs_col, parameter_col, type_col, value_col, lead_col, area_col   the thresholds table's columns (blank = find)
#   fw_label, res_label   how the type column names the two thresholds (default FW and RES)
#   from_date             only readings and warnings from this date (default 2023-01-01)
#   reset_hours           readings must stay below a threshold this long before it can be crossed again (default 6)
#   late_hours            a warning issued this long after a crossing is "late", not "missed" (default 24)
#   lookback_hours        how far back to look for the FW threshold crossing that triggered it (default 72)
#
# OUTPUT: config.TBL_FW_TIMING (restricted schema), one row per RES crossing or unmatched warning; overwritten in full.
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

import config
from utils.helpers import get_spark, utc_now, table_exists
from utils.telemetry_mapping import id_key
from utils.fw_timing import (prepare_thresholds, gauge_thresholds, choose_measure, validate_gauge, warning_issues,
                             summarise, RESULT_COLUMNS)

spark = get_spark()

for name, label in (("fffs_col", "FFFS_ID column"), ("parameter_col", "Parameter column"), ("type_col", "Threshold type column"),
                    ("value_col", "Threshold value column"), ("lead_col", "Target lead time (hours) column"),
                    ("area_col", "Warning area code column")):
    dbutils.widgets.text(name, "", label)
dbutils.widgets.text("fw_label", "FW", "Type value for the FW threshold")
dbutils.widgets.text("res_label", "RES", "Type value for the RES threshold")
dbutils.widgets.text("from_date", "2023-01-01", "From date")
dbutils.widgets.text("reset_hours", "6", "Reset hours")
dbutils.widgets.text("late_hours", "24", "Late hours")
dbutils.widgets.text("lookback_hours", "72", "Trigger look-back hours")

columns = {k: dbutils.widgets.get(w).strip() or None for k, w in (
    ("fffs_id", "fffs_col"), ("parameter", "parameter_col"), ("type", "type_col"), ("value", "value_col"), ("lead", "lead_col"), ("area", "area_col"))}
fw_label, res_label = dbutils.widgets.get("fw_label").strip(), dbutils.widgets.get("res_label").strip()
from_date = pd.Timestamp(dbutils.widgets.get("from_date").strip() or "2023-01-01")
reset_hours, late_hours, lookback_hours = (float(dbutils.widgets.get(w) or d) for w, d in (("reset_hours", 6), ("late_hours", 24), ("lookback_hours", 72)))

for t in (config.TBL_EA_THRESHOLDS, config.TBL_TELEMETRY_MAPPING, config.TBL_EA_RT_ALL, config.TBL_WARNING_PERIODS, config.TBL_WARNING_EPISODES):
    if not table_exists(spark, t):
        raise Exception(f"{t} does not exist. Run the job that builds it first.")


In [ ]:
# =============================================================================
# THRESHOLDS: read, find the columns, and show what was found
# =============================================================================

raw = spark.table(config.TBL_EA_THRESHOLDS).toPandas()
thr, trep = prepare_thresholds(raw, columns, fw_label, res_label)
print(f"Thresholds table: {trep['rows']:,} rows. Columns used: {trep['columns']}")
print(f"Values of the type column: {trep['type_values']}")
print(f"Dropped: {trep['dropped_no_kind']:,} rows whose type is neither '{fw_label}' nor '{res_label}', "
      f"{trep['dropped_no_value']:,} with no numeric value. Kept {len(thr):,}.")
gauges = gauge_thresholds(thr)
print(f"Gauges with a {res_label} threshold: {len(gauges):,}   with a target lead time: {int(gauges['target_lead_hours'].notna().sum()):,}   "
      f"with a warning area: {int(gauges['area_code'].notna().sum()):,}")
if gauges.empty:
    raise Exception(f"No gauge has a '{res_label}' threshold. Check the type column and the res_label widget (values found above).")
if gauges["area_code"].notna().sum() == 0:
    raise Exception("The thresholds table gives no warning area code, so a warning cannot be tied to a gauge. Name the column in area_col.")

# =============================================================================
# GAUGE -> MEASURE: the matched lines of the telemetry mapping, one measure per gauge and family
# =============================================================================

_have = spark.table(config.TBL_TELEMETRY_MAPPING).columns
mp = spark.table(config.TBL_TELEMETRY_MAPPING).where("match_status = 'matched'").select(
    "fffs_id", "telemetry_family", "code_family", "measure_uri", *[c for c in ("register_value_type", "n_readings") if c in _have]).toPandas()
mp["key"] = mp["fffs_id"].map(id_key)
mp["family"] = mp["telemetry_family"].fillna(mp["code_family"])       # the family each mapping row measures
measure = {}
for (key, fam), g in mp.groupby(["key", "family"], dropna=False):
    measure[(key, fam)] = choose_measure(g)
gauges["measure_uri"] = [measure.get((k, f)) or measure.get((k, None)) for k, f in zip(gauges["key"], gauges["family"])]
print(f"Gauges with a matched measure: {int(gauges['measure_uri'].notna().sum()):,} of {len(gauges):,}")
no_measure = gauges[gauges["measure_uri"].isna()]
if len(no_measure):
    print("No matched measure for (first 10):")
    print(no_measure[["fffs_id", "family", "area_code"]].head(10).to_string(index=False))
gauges = gauges[gauges["measure_uri"].notna()].reset_index(drop=True)


In [ ]:
# =============================================================================
# READINGS AND WARNINGS, THEN THE VALIDATION
# =============================================================================

uris = pd.DataFrame({"measure_uri": gauges["measure_uri"].unique()})
spark.createDataFrame(uris).createOrReplaceTempView("fw_timing_measures")
readings = spark.sql(f"""
    SELECT r.measure_uri, r.date_time, r.value
    FROM {config.TBL_EA_RT_ALL} r JOIN fw_timing_measures m ON m.measure_uri = r.measure_uri
    WHERE r.date_time >= '{from_date.date()}'
""").toPandas()
readings["date_time"] = pd.to_datetime(readings["date_time"]).dt.tz_localize(None) if getattr(pd.to_datetime(readings["date_time"]).dt, "tz", None) else pd.to_datetime(readings["date_time"])
print(f"Readings since {from_date.date()}: {len(readings):,} for {readings['measure_uri'].nunique():,} measures")

periods = spark.table(config.TBL_WARNING_PERIODS).toPandas()
episodes = spark.table(config.TBL_WARNING_EPISODES).toPandas()
issues = warning_issues(periods, episodes, as_of=utc_now().replace(tzinfo=None))
issues = issues[issues["issue_time"] >= from_date]
print(f"Flood-warning-level episodes since {from_date.date()}: {len(issues):,}")

by_measure = {u: g[["date_time", "value"]] for u, g in readings.groupby("measure_uri")}
rows = []
for gauge in gauges.to_dict("records"):
    series = by_measure.get(gauge["measure_uri"])
    if series is None or gauge["area_code"] is None:
        continue
    area_issues = issues[issues["ea_area_code"] == gauge["area_code"]]
    for r in validate_gauge(gauge, series, area_issues, reset_hours, late_hours, lookback_hours):
        r["measure_uri"] = gauge["measure_uri"]
        rows.append(r)
results = pd.DataFrame(rows, columns=RESULT_COLUMNS + ["measure_uri"])
print(f"Result rows: {len(results):,}")
summary = summarise(results)
print(summary.to_string(index=False))
display(summary)


In [ ]:
# =============================================================================
# WRITE: restricted schema, overwritten in full
# =============================================================================

SCHEMA = StructType([StructField(c, DoubleType() if c in ("lead_hours", "target_lead_hours", "issue_delay_hours", "res_value", "fw_value") else StringType(), True)
                     for c in RESULT_COLUMNS + ["measure_uri"]] + [StructField("computed_at", StringType(), True)])
out = results.copy()
for c in ("res_time", "issue_time", "trigger_time"):
    out[c] = pd.to_datetime(out[c]).dt.strftime("%Y-%m-%dT%H:%M:%S")
out["computed_at"] = utc_now().isoformat()
cols = [f.name for f in SCHEMA.fields]
out = out[cols].astype(object).where(out[cols].notna(), None)
(
    spark.createDataFrame(out, SCHEMA).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.TBL_FW_TIMING)
)
spark.sql(f"""
    COMMENT ON TABLE {config.TBL_FW_TIMING} IS
    'Flood warning timing validation (27_fw_timing_validation). One row per upward crossing of a gauge RES threshold (kind res_event:
     outcome on time, too short, late or missed, with lead_hours against target_lead_hours) or per flood warning with no crossing
     (kind warning_only: RES not reached, or no readings). trigger_time is the FW threshold crossing before it and issue_delay_hours the
     time the warning took after it. Restricted: it is built from the thresholds.'
""")
print(f"Written {spark.table(config.TBL_FW_TIMING).count():,} rows to {config.TBL_FW_TIMING}")
dbutils.notebook.exit("success")
